# IsoUMI quick start

This notebook verifies the bundled IsoUMI and samtools executables, runs the minimal fixture, and inspects corrected UMIs and duplicate flags.

In [ ]:
import subprocess

subprocess.run(["isoumi", "--version"], check=True)
subprocess.run(["samtools", "--version"], check=True)

## Run the bundled example

The relaxed ratio in this tiny fixture makes `AAAT` collapse into the higher-count `AAAA` molecule.

In [ ]:
from pathlib import Path
import shutil

work = Path("/workspace/demo")
if work.exists():
    shutil.rmtree(work)
work.mkdir(parents=True)

cmd = [
    "isoumi",
    "--bam", "/opt/isoumi/examples/minimal.sam",
    "--out", str(work / "minimal"),
    "--threads", "1",
    "--buckets", "4",
    "--ham", "1",
    "--ratio", "0.60",
    "--emit-tsv",
    "--emit-explain",
]
subprocess.run(cmd, check=True)
sorted(path.name for path in work.iterdir())

In [ ]:
print((work / "minimal.corrections.tsv").read_text())
view = subprocess.run(
    ["samtools", "view", str(work / "minimal.dedup.bam")],
    check=True, text=True, capture_output=True,
).stdout
print(view)

## Export representative reads only

IsoUMI retains every read and marks duplicates with SAM flag `0x400`. Filtering with `-F 1024` physically removes marked duplicates.

In [ ]:
representatives = work / "minimal.representatives.bam"
with representatives.open("wb") as output_handle:
    subprocess.run(
        ["samtools", "view", "-b", "-F", "1024", str(work / "minimal.dedup.bam")],
        check=True, stdout=output_handle,
    )
subprocess.run(["samtools", "view", "-c", str(representatives)], check=True)